# cfos-mapper from Python

The same steps as the command line, as plain functions: `import cfos_mapper as cf`. Every function works on
one brain (or one test region of a brain) and takes the `settings`. Positions are always (z, y, x) in
full-resolution voxels.

In [ ]:
import os

# the settings file of the project (where the brains are, where results go, all parameters)
os.environ["CFOS_MAPPER_SETTINGS"] = "/mif-users/no_backup/Gritti_Nicola/IDIBAPS-brains/code/settings.json"

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cfos_mapper as cf

settings = cf.load_settings()
cf.brain_names(settings)

## 1. A brain (pymif, nothing is loaded until needed)

In [ ]:
BRAIN = "O15"
brain = cf.open_brain(BRAIN, settings)
print(brain.metadata["channel_names"], [d.shape for d in brain.data][:3], cf.voxel_size(brain))

lowest = len(brain.data) - 1
cfos = np.asarray(cf.get_channel(brain, settings["signal_channel"], lowest))
plt.imshow(cfos[cfos.shape[0] // 2], cmap="gray", vmax=np.percentile(cfos, 99.5))
plt.title(f"{BRAIN}, middle plane (lowest resolution)");

## 2. A test region and its candidates

In [ ]:
REGION = "regionA_cortex"
signal, background, size, candidates = cf.prepare_region(BRAIN, REGION, settings)
print(f"{len(candidates)} candidates in {signal.shape} voxels")

z0, z1 = 140, 160
shown = candidates[(candidates.z >= z0) & (candidates.z < z1)]
plt.figure(figsize=(7, 7))
plt.imshow(signal[z0:z1].max(0), cmap="gray", vmax=np.percentile(signal, 99.9))
plt.scatter(shown.x, shown.y, s=80, facecolors="none", edgecolors="yellow")
plt.title(f"candidates in planes {z0}-{z1}");

## 3. Your annotations, and what a network says

In [ ]:
annotations = cf.load_annotations(cf.annotations_file(BRAIN, REGION, settings))
print(annotations.label.value_counts())

networks = sorted(p.parent.name for p in Path(settings["networks_folder"]).glob("*/network.keras"))
NETWORK = networks[-1] if networks else "pretrained"
predictions = cf.get_predictions(BRAIN, REGION, NETWORK, settings)   # GPU the first time, then from file

plain_tissue = cf.far_from_candidates(predictions["control"], candidates, size)
trained_on = cf.used_in_training(annotations, Path(settings["networks_folder"]) / NETWORK, BRAIN, REGION)
pd.Series(cf.prediction_statistics(predictions["calls"], annotations, plain_tissue, trained_on), name=NETWORK)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
cf.draw_calls(ax, signal, predictions["calls"], f"{NETWORK}: {BRAIN} {REGION}")

## 4. Train and run on a whole brain

Both take long and use the GPU: run them in `tmux` or as cluster jobs (command line). In Python:
```python
import os; os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # before cfos_mapper uses the GPU
cf.train_network("network_v3", settings, epochs=150)
cf.run_brain("O15", "network_v3", settings, planes=(900, 1000))   # quick test
```
Here we look at a finished whole-brain run:

In [ ]:
runs = sorted(p.parent.name for p in (Path(settings["results_folder"]) / BRAIN / "full_brain").glob("*/cells.csv"))
RUN = runs[-1]
cells = pd.read_csv(Path(settings["results_folder"]) / BRAIN / "full_brain" / RUN / "cells.csv")
print(f"{RUN}: {len(cells):,} cells")

## 5. Atlas registration (brainreg) and cells per region

In [ ]:
if not cf.registration_ready(BRAIN, settings):
    cf.register_brain(BRAIN, settings)   # ~30-60 min on the CPU

import tifffile
folder = cf.registration_folder(BRAIN, settings)
brain_small = tifffile.imread(folder / "downsampled.tiff")        # the brain, atlas orientation and resolution
atlas_on_brain = tifffile.imread(folder / "registered_atlas.tiff")
plane = brain_small.shape[0] // 2
plt.figure(figsize=(7, 6))
plt.imshow(brain_small[plane], cmap="gray", vmax=np.percentile(brain_small, 99.5))
plt.contour(atlas_on_brain[plane], levels=np.unique(atlas_on_brain[plane]), colors="orange", linewidths=0.3)
plt.title("check: atlas outlines on the brain (coronal)");

In [ ]:
table = cf.count_cells_per_region(BRAIN, RUN, settings, run=RUN)
regions = table[(table.depth >= 6) & (table.volume_mm3 > 0.1)].nlargest(20, "cells_per_mm3")
plt.figure(figsize=(6, 6))
plt.barh(regions.acronym[::-1], regions.cells_per_mm3[::-1], color="#2a78d6")
plt.xlabel("cells per mm³")
plt.title(f"{BRAIN}: densest regions");

In [ ]:
# all brains with this run, side by side
cf.summary_table(RUN, settings, run=RUN).head(20)